In [ ]:
# import IPython
# IPython.Application.instance().kernel.do_shutdown(True) # Reset kernel

In [ ]:
!pip install vllm==0.10.0
!pip install triton==3.2.0
!pip install langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai
!pip uninstall -y openai
!pip install openai==1.90.0

In [ ]:
# Install pyngrok
!pip install pyngrok

# Import and setup ngrok
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
NGROK_KEY = user_secrets.get_secret("NGROK_KEY_2")

# Set ngrok authtoken
ngrok.set_auth_token(NGROK_KEY)

In [ ]:
import os
BRAVE_API_KEY = user_secrets.get_secret("BRAVE_API_KEY")
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")

os.environ["WEB_SEARCH_SSL"] = str(False)
os.environ["GOOGLE_SEARCH_CX"] = "9501a956284f141ab"
os.environ["GOOGLE_SEARCH_API_KEY"] = GOOGLE_API_KEY
os.environ["BRAVE_SEARCH_API_KEY"] = BRAVE_API_KEY

DOMAIN = "https://dbb932f78d1f.ngrok-free.app"
NGROK_PORT = 8002


In [ ]:
# Start ngrok tunnel using pyngrok
tunnel = ngrok.connect(NGROK_PORT, "http")
public_url = tunnel.public_url

print(f"Ngrok tunnel started!")
print(f"Public URL: {public_url}")

In [ ]:
import requests
import io
import tarfile
import shutil
def unpack(data: bytes, path: str):
    if os.path.exists(path):
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_p(name: str):
    unpack(requests.get(f"{DOMAIN}/script/{name}").content, name)
def unpack_pl(*names: str):
    for name in names:
        unpack_p(name)
unpack_pl(
    "vllm_worker", "web_search", "server"
)

In [ ]:
from vllm_worker import prepare
from server import construct_app, CustomQA, create_client_info, set_active, PRELOAD_MODEL, KaggleRequest
from typing import AsyncGenerator
import uvicorn
prepare()

In [ ]:
# Client info với domain sẽ tự update khi chạy với ngrok
CLIENT_INFO = create_client_info("http://127.0.0.1:8002")

In [ ]:
import threading
def thread_task():
    ws_pipeline = CustomQA(set_active)
    REQUEST_STORAGE: dict[str, tuple[str, KaggleRequest]] = {}
    
    async def pre_inference_function(request: KaggleRequest):
        prompt, pre_output = await ws_pipeline.pre_inference(
            request["model_id"],
            request["question"],
            request["stream_id"],
            request["params"],
            request.get("vector_sources"),
            request.get("web_search_keywords")
        )
        REQUEST_STORAGE[request["stream_id"]] = (prompt, request)
        return pre_output
    
    async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
        prompt, request = REQUEST_STORAGE.pop(stream_id)
        generator = await ws_pipeline.inference(prompt, request)
        return generator
    
    app = construct_app(
        server_domain=DOMAIN,
        info=CLIENT_INFO,
        pre_inference=pre_inference_function,
        inferece=inference_function,
        init_tasks=[
            ws_pipeline.start(),
            ws_pipeline.preload(PRELOAD_MODEL)
        ]
    )
    
    uvicorn.run(app, port=NGROK_PORT)

thread = threading.Thread(target=thread_task, daemon=True)
thread.start()